# Junção de tabelas:

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np



## 1. Presença por Deputado:

### **Métrica necessária**

- Número total de sessões deliberativas (por deputado)

- Número de presenças (por deputado)

- Taxa de presença (%)

| Dataset      | Coluna          | Para quê?                                                           |
| ------------ | --------------- | ------------------------------------------------------------------- |
| df_eventos   | `id`            | identificar cada evento (sessão)                                    |
| df_eventos   | `descricaoTipo` | filtrar somente “Sessão Deliberativa”                               |
| df_presencas | `id_evento`     | ligar presenças aos eventos                                         |
| df_presencas | `id_deputado`   | identificar o deputado                                              |
| df_presencas | `tipo_presenca` | classificar se foi **Presença**, **Ausência**, **Justificada** etc. |

### **Derivadas**

- total_sessoes

- total_presencas

- taxa_presenca = total_presencas / total_sessoes

### Criar colunas:

## 2. Sessões por Taxa de Votações Participadas:

### **Métricas necessárias**

- Sessões totais

- Votações possíveis

- Votações participadas

- Taxa de votação por deputado

| Dataset      | Coluna        | Para quê?                                         |
| ------------ | ------------- | ------------------------------------------------- |
| votacoes_all | `id`          | id da votação                                     |
| votacoes_all | `uriEvento`   | para saber qual evento originou a votação         |
| df_eventos   | `id`          | conectar votação ao evento                        |
| df_presencas | `id_deputado` | identificar deputado                              |
| df_presencas | `id_evento`   | saber em quais eventos o deputado esteve presente |

### **Derivadas**

- votacoes_possiveis = total de votações nos eventos que o deputado participou

- votacoes_participadas = quantidade de votações registradas como participação
(→ isso depende se você tem o dataset de votos por deputado. Se não tiver, você terá que assumir participação quando presença = presente.)

- taxa_votacao = votacoes_participadas / votacoes_possiveis


### Criar colunas:

## 3. Faltas Injustificadas vs Limite CLT:

### **Métrica necessária**

- Faltas contabilizadas como não justificadas

- Comparação com limite CLT (12 faltas por ano)

| Dataset      | Coluna          | Para quê?                             |
| ------------ | --------------- | ------------------------------------- |
| df_presencas | `tipo_presenca` | detectar “ausência sem justificativa” |
| df_presencas | `id_deputado`   | identificar deputado                  |

### **Derivadas**

- faltas_injustificadas = count(tipo_presenca == 'Ausência')

- (opcional) faltas_justificadas = count(tipo_presenca == 'Justificada')

### Criar colunas:

## 4. Ganho por Dia Trabalhado e por Votação:

### **Alinhar** 
id_deputado, salario_bruto


- ganho_por_dia_trabalhado = salario_bruto / total_presencas

- ganho_por_votacao_participada = salario_bruto / votacoes_participadas

### Criar colunas:

## 5. Desigualdade — Deputados VS Trabalhadores CLT:

- O dataset dos deputados (já acima)

- Um dataset CLT fictício ou real

- Derivada: ganho_por_dia_trabalhado (já calculada)

### Criar colunas:

In [2]:
# Presenças em sessões — coleta total por ano
import io
import requests
import pandas as pd
from pathlib import Path
from datetime import date

# salva na MESMA pasta do código
DATA_DIR = Path(".")
OUT = DATA_DIR / "presencas.csv"

BASE_ARQ = (
    "http://dadosabertos.camara.leg.br/arquivos/"
    "eventosPresencaDeputados/csv/eventosPresencaDeputados-{ano}.csv"
)

# Anos a cobrir
anos = list(range(2020, date.today().year + 1))


def padronizar_presencas(df_raw: pd.DataFrame, ano: int) -> pd.DataFrame:
    # normaliza nomes
    cols_lower = {c.lower(): c for c in df_raw.columns}

    # coluna do id do evento
    id_evento_col = None
    for key in ("idevento", "id_evento", "evento_id", "id"):
        if key in cols_lower:
            id_evento_col = cols_lower[key]
            break
    if not id_evento_col:
        id_evento_col = next(
            (c for c in df_raw.columns if "evento" in c.lower()), None
        )
    if not id_evento_col:
        raise KeyError(
            f"[{ano}] Não achei coluna de evento. Colunas: {list(df_raw.columns)}"
        )

    # coluna do id do deputado
    id_dep_col = None
    for key in ("iddeputado", "id_deputado", "idparlamentar", "idecadastro"):
        if key in cols_lower:
            id_dep_col = cols_lower[key]
            break
    if not id_dep_col:
        id_dep_col = next(
            (c for c in df_raw.columns if "deputad" in c.lower()), None
        )

    # coluna do tipo de presença
    tipo_col = None
    for key in ("tipopresenca", "tipo_presenca"):
        if key in cols_lower:
            tipo_col = cols_lower[key]
            break
    if not tipo_col:
        tipo_col = next(
            (
                c
                for c in df_raw.columns
                if "presen" in c.lower() and "tipo" in c.lower()
            ),
            None,
        )

    out = pd.DataFrame()

    # id do evento
    out["id_evento"] = pd.to_numeric(
        df_raw[id_evento_col].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce",
    ).astype("Int64")

    # id do deputado (se existir)
    out["id_deputado"] = (
        pd.to_numeric(
            df_raw[id_dep_col].astype(str).str.extract(r"(\d+)")[0],
            errors="coerce",
        ).astype("Int64")
        if id_dep_col
        else pd.NA
    )

    # tipo de presença (se existir)
    out["tipo_presenca"] = df_raw[tipo_col] if tipo_col else pd.NA
    out["ano_origem"] = ano

    out = out.dropna(subset=["id_evento"])
    return out


parts = []

for ano in anos:
    url = BASE_ARQ.format(ano=ano)
    print(f"Baixando {ano} de {url}...")
    r = requests.get(url, timeout=60)

    if r.status_code == 404:
        print(f"  {ano}: arquivo não encontrado (404), pulando.")
        continue

    r.raise_for_status()
    buf = io.BytesIO(r.content)

    # CSVs do portal costumam usar ';' e encoding UTF-8
    df_raw = pd.read_csv(buf, sep=";", dtype=str, low_memory=False)

    try:
        df_pad = padronizar_presencas(df_raw, ano)
        parts.append(df_pad)
        print(f"  {ano}: OK, linhas = {len(df_pad)}")
    except Exception as e:
        print(f"  {ano}: ERRO ao padronizar -> {e}")

presencas = (
    pd.concat(parts, ignore_index=True)
    if parts
    else pd.DataFrame(
        columns=["id_evento", "id_deputado", "tipo_presenca", "ano_origem"]
    )
)

presencas = presencas.drop_duplicates(
    subset=["id_evento", "id_deputado", "tipo_presenca"]
)

presencas.to_csv(OUT, index=False)
print(f"OK — presenças salvas em {OUT.resolve()} | linhas: {len(presencas)}")


Baixando 2020 de http://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2020.csv...
  2020: OK, linhas = 68448
Baixando 2021 de http://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2021.csv...
  2021: OK, linhas = 146387
Baixando 2022 de http://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2022.csv...
  2022: OK, linhas = 75734
Baixando 2023 de http://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2023.csv...
  2023: OK, linhas = 107754
Baixando 2024 de http://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2024.csv...
  2024: OK, linhas = 77637
Baixando 2025 de http://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2025.csv...
  2025: OK, linhas = 99018
OK — presenças salvas em /home/nemo/puc/tde_coleta/TDE-Coleta/lab/transform/pres